# EXP-15: Multi-Seed + Multi-C Optimized Ensemble

**Competition:** SPR 2026 — Mammography Report Classification  
**Base:** EXP-13 (Multi-Seed Three-SVC) + Clinical Geometry insights  
**Enhancements over EXP-13:**
- Varia C do LinearSVC por seed (0.5, 0.8, 1.0, 1.5, 2.0) → diversidade real
- LGBs com hiperparâmetros diferentes por seed
- Busca de pesos ótimos via OOF (Dirichlet sampling, 20k combinações)
- Threshold search para TODAS as 7 classes (0-6) — organizer avisou sobre classes raras
- GroupKFold 5-fold CV para validação local confiável
**Runtime:** CPU only

In [ ]:
import os, re, hashlib, warnings, time
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import FeatureUnion
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, classification_report
from scipy.sparse import hstack, csr_matrix
import lightgbm as lgb
warnings.filterwarnings('ignore')

TRAIN_PATH = '/kaggle/input/competitions/spr-2026-mammography-report-classification/train.csv'
TEST_PATH  = '/kaggle/input/competitions/spr-2026-mammography-report-classification/test.csv'

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

N_FOLDS = 5
N_CLASSES = 7
SEEDS = [42, 123, 456, 789, 2026]

print(f'Train: {train.shape}, Test: {test.shape}')
print(f'Class distribution:\n{train["target"].value_counts().sort_index()}')

In [ ]:
def clean_achados(s: str) -> str:
    if pd.isna(s): return ""
    s = str(s).strip().lower()
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{2,}", "\n", s)
    match = re.search(r'achados:(.*?)(análise comparativa:|$)', s, flags=re.DOTALL)
    if match: s = match.group(1).strip()
    s = re.sub(r'[0-9]+,[0-9]+', 'NUM', s)
    s = re.sub(r'[0-9]+', 'NUM', s)
    return s

def clean_full(s: str) -> str:
    if pd.isna(s): return ""
    s = str(s).strip().lower()
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r'[0-9]+,[0-9]+', 'NUM', s)
    s = re.sub(r'[0-9]+', 'NUM', s)
    return s

def stable_hash(s: str) -> str:
    return hashlib.md5(s.encode("utf-8")).hexdigest()

def extract_dense_features(df):
    features = pd.DataFrame(index=df.index)
    text_col = df['report'].fillna('').astype(str).str.lower()
    features['report_length']   = text_col.apply(len)
    features['has_measurement'] = text_col.str.contains(r'\b(?:cm|mm|medindo)\b', regex=True).astype(int)
    features['has_spiculation'] = text_col.str.contains(r'espiculad', regex=True).astype(int)
    features['has_distortion']  = text_col.str.contains(r'distorção arquitetural', regex=True).astype(int)
    features['has_biopsy']      = text_col.str.contains(r'biopsy|biópsia|resultado de cine|carcinoma', regex=True).astype(int)
    return csr_matrix(features.values)

train["achados"] = train["report"].apply(clean_achados)
train["full"]    = train["report"].apply(clean_full)
test["achados"]  = test["report"].apply(clean_achados)
test["full"]     = test["report"].apply(clean_full)
train["group"]   = train["report"].apply(stable_hash)

X_train_dense = extract_dense_features(train)
X_test_dense  = extract_dense_features(test)
y = train["target"].astype(int).values
groups = train["group"].values

print("Preprocessing done.")

In [ ]:
print("Building TF-IDF features...")

tfidf_A = FeatureUnion([
    ("word", TfidfVectorizer(ngram_range=(1, 3), min_df=3, max_df=0.95, sublinear_tf=True)),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_df=0.95,
                             sublinear_tf=True, max_features=80000))
])
X_train_A = tfidf_A.fit_transform(train["achados"].values)
X_test_A  = tfidf_A.transform(test["achados"].values)

tfidf_F = FeatureUnion([
    ("word", TfidfVectorizer(ngram_range=(1, 3), min_df=3, max_df=0.95, sublinear_tf=True)),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_df=0.95,
                             sublinear_tf=True, max_features=80000))
])
X_train_F = tfidf_F.fit_transform(train["full"].values)
X_test_F  = tfidf_F.transform(test["full"].values)

tfidf_F2 = FeatureUnion([
    ("word", TfidfVectorizer(ngram_range=(1, 3), min_df=3, max_df=0.95, sublinear_tf=True)),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 6), min_df=3, max_df=0.95,
                             sublinear_tf=True, max_features=100000))
])
X_train_F2 = tfidf_F2.fit_transform(train["full"].values)
X_test_F2  = tfidf_F2.transform(test["full"].values)

X_train_lgb = hstack([X_train_A, X_train_dense]).tocsr()
X_test_lgb  = hstack([X_test_A,  X_test_dense]).tocsr()

print(f"SVC-A: {X_train_A.shape[1]}, SVC-F: {X_train_F.shape[1]}, SVC-F2: {X_train_F2.shape[1]}, LGB: {X_train_lgb.shape[1]}")

In [ ]:
# =============================================================================
# Multi-Seed + Multi-C GroupKFold CV — diversidade real entre modelos
# Varia seed E hiperparâmetros (C para SVC, params para LGB)
# =============================================================================

gkf = GroupKFold(n_splits=N_FOLDS)
folds = list(gkf.split(X_train_A, y, groups))

oof_probas = {}   # model_name -> (n_train, 7)
test_probas = {}  # model_name -> (n_test, 7)

# Configs: (seed, C_value) para SVCs — diversidade de hiperparâmetros
SVC_CONFIGS = [
    (42, 1.0), (123, 0.8), (456, 1.5), (789, 0.5), (2026, 2.0)
]

# Configs para LGB — diferentes hiperparâmetros por seed
LGB_CONFIGS = [
    (42,  dict(n_estimators=300, learning_rate=0.05, max_depth=6)),
    (123, dict(n_estimators=400, learning_rate=0.03, max_depth=7, num_leaves=63)),
    (456, dict(n_estimators=500, learning_rate=0.02, max_depth=8, min_child_samples=30)),
    (789, dict(n_estimators=300, learning_rate=0.05, max_depth=5, num_leaves=31)),
    (2026, dict(n_estimators=350, learning_rate=0.04, max_depth=6, subsample=0.8, colsample_bytree=0.7)),
]

t0 = time.time()

for seed, C_val in SVC_CONFIGS:
    for feat_name, X_tr_feat, X_te_feat in [('A', X_train_A, X_test_A),
                                              ('F', X_train_F, X_test_F),
                                              ('F2', X_train_F2, X_test_F2)]:
        name = f'svc_{feat_name}_s{seed}_C{C_val}'
        oof = np.zeros((len(train), N_CLASSES), dtype=np.float64)
        te_acc = np.zeros((len(test), N_CLASSES), dtype=np.float64)
        for fi, (tr_idx, va_idx) in enumerate(folds):
            m = CalibratedClassifierCV(
                LinearSVC(class_weight="balanced", random_state=seed, max_iter=10000, C=C_val),
                cv=3, method='sigmoid'
            )
            m.fit(X_tr_feat[tr_idx], y[tr_idx])
            oof[va_idx] = m.predict_proba(X_tr_feat[va_idx])
            te_acc += m.predict_proba(X_te_feat) / N_FOLDS
        oof_f1 = f1_score(y, np.argmax(oof, axis=1), average='macro')
        print(f"{name}: OOF F1={oof_f1:.4f}")
        oof_probas[name] = oof
        test_probas[name] = te_acc

for seed, params in LGB_CONFIGS:
    name = f'lgb_s{seed}'
    oof = np.zeros((len(train), N_CLASSES), dtype=np.float64)
    te_acc = np.zeros((len(test), N_CLASSES), dtype=np.float64)
    for fi, (tr_idx, va_idx) in enumerate(folds):
        m = lgb.LGBMClassifier(class_weight='balanced', random_state=seed, n_jobs=-1, verbose=-1, **params)
        m.fit(X_train_lgb[tr_idx], y[tr_idx])
        oof[va_idx] = m.predict_proba(X_train_lgb[va_idx])
        te_acc += m.predict_proba(X_test_lgb) / N_FOLDS
    oof_f1 = f1_score(y, np.argmax(oof, axis=1), average='macro')
    print(f"{name}: OOF F1={oof_f1:.4f}")
    oof_probas[name] = oof
    test_probas[name] = te_acc

elapsed = time.time() - t0
print(f"\n{'='*60}")
print(f"All {len(oof_probas)} models trained in {elapsed:.0f}s")

In [ ]:
# =============================================================================
# Ensemble: average por tipo + busca de pesos ótimos via OOF
# =============================================================================

def avg_across_configs(probas_dict, prefix):
    matching = {k: v for k, v in probas_dict.items() if k.startswith(prefix)}
    if not matching:
        return None
    return np.mean(list(matching.values()), axis=0)

# Média das probabilidades por tipo de modelo
oof_by_type = {}
te_by_type = {}
for prefix, label in [('svc_A', 'svc_A'), ('svc_F_', 'svc_F'), ('svc_F2', 'svc_F2'), ('lgb', 'lgb')]:
    oof_by_type[label] = avg_across_configs(oof_probas, prefix)
    te_by_type[label]  = avg_across_configs(test_probas, prefix)

# Mostrar score individual de cada tipo
for label, oof in oof_by_type.items():
    if oof is not None:
        f1 = f1_score(y, np.argmax(oof, axis=1), average='macro')
        print(f"  {label} (averaged): OOF F1={f1:.4f}")

# --- Busca de pesos ótimos via OOF (grid search) ---
print("\nSearching optimal weights (OOF)...")
types = ['svc_A', 'svc_F', 'svc_F2', 'lgb']
best_f1 = 0
best_weights = None

np.random.seed(42)
# Dirichlet sampling: 20k combinações aleatórias
for _ in range(20000):
    raw = np.random.dirichlet(np.ones(len(types)))
    w = np.round(raw / 0.05) * 0.05
    if w.sum() == 0:
        continue
    w = w / w.sum()
    
    blend = sum(w[i] * oof_by_type[types[i]] for i in range(len(types)))
    preds = np.argmax(blend, axis=1)
    score = f1_score(y, preds, average='macro')
    
    if score > best_f1:
        best_f1 = score
        best_weights = w.copy()

print(f"\nBest weights (OOF F1={best_f1:.4f}):")
for t, w in zip(types, best_weights):
    print(f"  {t}: {w:.2f}")

# Comparar com pesos fixos do V12
oof_v12 = 0.70 * (0.25 * oof_by_type['svc_A'] + 0.40 * oof_by_type['svc_F'] + 0.35 * oof_by_type['svc_F2']) + 0.30 * oof_by_type['lgb']
v12_f1 = f1_score(y, np.argmax(oof_v12, axis=1), average='macro')
print(f"\nV12 fixed weights OOF F1: {v12_f1:.4f}")

# Usar os melhores pesos
oof_blend = sum(best_weights[i] * oof_by_type[types[i]] for i in range(len(types)))
te_blend  = sum(best_weights[i] * te_by_type[types[i]] for i in range(len(types)))

baseline_preds = np.argmax(oof_blend, axis=1)
baseline_f1 = f1_score(y, baseline_preds, average='macro')
print(f"\nOOF macro-F1 (optimal weights, no thresholds): {baseline_f1:.4f}")
print(f"\nPer-class OOF F1:")
print(classification_report(y, baseline_preds, digits=4))

In [ ]:
# =============================================================================
# OOF-Optimized Threshold Search — TODAS as classes (0-6)
# O organizador avisou: classes raras impactam muito o macro F1
# =============================================================================

# Prioridade: classes mais severas primeiro, depois as menores
threshold_classes = [6, 5, 4, 3, 1, 0]
threshold_range = np.arange(0.05, 0.55, 0.01)

def apply_thresholds(proba, thresholds):
    """Aplica thresholds com prioridade hierárquica."""
    preds = np.argmax(proba, axis=1).copy()
    for cls in threshold_classes:
        if cls in thresholds:
            mask = proba[:, cls] > thresholds[cls]
            # Não sobreescrever classes de prioridade mais alta
            for higher_cls in threshold_classes:
                if higher_cls == cls:
                    break
                if higher_cls in thresholds:
                    mask = mask & (preds != higher_cls)
            preds[mask] = cls
    return preds

best_thresholds = {}
current_f1 = baseline_f1

for cls in threshold_classes:
    best_cls_f1 = current_f1
    best_cls_thr = None
    for thr in threshold_range:
        trial = {**best_thresholds, cls: thr}
        trial_preds = apply_thresholds(oof_blend, trial)
        trial_f1 = f1_score(y, trial_preds, average='macro')
        if trial_f1 > best_cls_f1:
            best_cls_f1 = trial_f1
            best_cls_thr = thr
    if best_cls_thr is not None:
        best_thresholds[cls] = best_cls_thr
        current_f1 = best_cls_f1
        print(f"Class {cls}: threshold={best_cls_thr:.2f}, macro-F1={best_cls_f1:.4f}")
    else:
        print(f"Class {cls}: no improvement from thresholds")

final_oof_preds = apply_thresholds(oof_blend, best_thresholds)
final_oof_f1 = f1_score(y, final_oof_preds, average='macro')
print(f"\nFinal OOF macro-F1 with thresholds: {final_oof_f1:.4f}")
print(f"Optimal thresholds: {best_thresholds}")
print(f"\nPer-class F1 with thresholds:")
print(classification_report(y, final_oof_preds, digits=4))

In [ ]:
# =============================================================================
# Apply thresholds + clinical guardrails → submission
# =============================================================================

test_preds = apply_thresholds(te_blend, best_thresholds)

def apply_safe_rules(row):
    text = str(row['report']).lower()
    current_pred = int(row['target'])
    if re.search(r'resultado de cine grau 3|carcinoma|\bcdis\b', text):
        return 6
    if 'espiculad' in text and 'distorção' in text and current_pred < 4:
        return 5
    return current_pred

test['target'] = test_preds
test['target'] = test.apply(apply_safe_rules, axis=1)

submission = test[['ID', 'target']].copy()
submission['target'] = submission['target'].astype(int)
submission.to_csv('submission.csv', index=False)

print('Prediction distribution:')
print(submission['target'].value_counts().sort_index())
print(f'\nSubmission saved! Shape: {submission.shape}')
print(submission)